# LangChain + LangGraph for AI/ML Model Serving

This notebook uses the same architecture as the AI/ML observability stack and focuses on the division of responsibility:

- **LangChain**: models, prompts, tools, structured output and integrations.
- **LangGraph**: state, nodes, conditional routing, loops and checkpoints.
- **Pydantic**: validated contracts between probabilistic reasoning and deterministic execution.
- **FastAPI / MCP**: expose Python capabilities to applications and agents.
- **vLLM**: optional OpenAI-compatible LLM serving backend.
- **MLflow**: model registry/training/serving boundary.
- **PostgreSQL + pgvector**: persistent structured/semantic knowledge.
- **Prometheus / OpenTelemetry / Langfuse**: operational and AI observability.

The first half is completely local and deterministic so you can understand the graph before connecting a real LLM.


## 1. The mental model

A normal ML training pipeline is usually a DAG:

`prepare -> train -> evaluate -> register`

LangGraph can wrap that DAG with a decision loop:

`inspect -> decide -> run pipeline -> evaluate -> retry/adjust OR promote`

The ML DAG executes a known recipe. **LangGraph decides what should happen next.**


In [ ]:
# Optional installation cell. Uncomment in a fresh notebook environment.
# %pip install -q "langgraph>=1.0" "langchain>=1.0" "langchain-openai>=1.0" #     "pydantic>=2" "fastapi>=0.115" "prometheus-client>=0.21" "mlflow>=3.0"


## 2. Pydantic turns an agent decision into a contract

Do not route production workflows on free-form LLM prose. Validate the decision first.


In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class TrainingDecision(BaseModel):
    action: Literal["train", "retry", "promote", "human_review", "stop"]
    model_family: Literal["xgboost", "sklearn", "tensorflow", "pytorch"]
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str
    max_trials: int = Field(default=3, ge=1, le=20)

example = TrainingDecision(
    action="retry",
    model_family="xgboost",
    confidence=0.91,
    reason="Candidate F1 did not exceed the champion.",
)
example


## 3. Python functions are the capabilities

Keep business/model logic independent of LangGraph. The same function can later be exposed through FastAPI, MCP, Kafka consumers, tests, or a graph node.


In [ ]:
MODEL_HISTORY = {
    "xgboost": {"champion_f1": 0.92, "last_candidate_f1": 0.89},
    "sklearn": {"champion_f1": 0.88, "last_candidate_f1": 0.90},
}

def get_model_health(model_name: str) -> dict:
    return {"model": model_name, "status": "healthy", "version": "demo-v1"}

def get_model_metrics(model_name: str) -> dict:
    return MODEL_HISTORY.get(model_name, {"champion_f1": 0.0, "last_candidate_f1": 0.0})

def launch_training(model_name: str, attempt: int) -> dict:
    # Deterministic demo replacing an MLflow/KFP training job.
    candidate = min(0.86 + 0.03 * attempt, 0.97)
    return {"model": model_name, "attempt": attempt, "candidate_f1": candidate}

get_model_health("xgboost"), get_model_metrics("xgboost")


### The exact same function exposed with FastAPI

This is the pattern that makes FastAPI so compelling for model serving.


In [ ]:
# Illustration only; running a web server is not required for the graph demo.
from fastapi import FastAPI

app = FastAPI()

@app.get("/models/{model_name}/health")
def model_health(model_name: str):
    return get_model_health(model_name)

# get_model_health() remains ordinary Python business logic.


## 4. LangGraph state

State is the typed object passed between graph nodes. Nodes return only the fields they change.


In [ ]:
from typing_extensions import TypedDict

class ServingState(TypedDict, total=False):
    request: str
    model_name: str
    attempt: int
    candidate_f1: float
    champion_f1: float
    decision: TrainingDecision
    history: list[str]

initial_state: ServingState = {
    "request": "Improve the churn classifier if the challenger beats production.",
    "model_name": "xgboost",
    "attempt": 0,
    "history": [],
}


## 5. Nodes are just Python functions

This is one of the most important LangGraph ideas: a node does not need to be an LLM call. It can be SQL, MLflow, pandas, Prometheus, an MCP tool, a REST call, or ordinary Python.


In [ ]:
def inspect_model(state: ServingState):
    metrics = get_model_metrics(state["model_name"])
    return {
        "champion_f1": metrics["champion_f1"],
        "history": state.get("history", []) + ["inspected production model"],
    }

def train_candidate(state: ServingState):
    attempt = state.get("attempt", 0) + 1
    result = launch_training(state["model_name"], attempt)
    return {
        "attempt": attempt,
        "candidate_f1": result["candidate_f1"],
        "history": state.get("history", []) + [f"trained attempt {attempt}: F1={result['candidate_f1']:.3f}"],
    }

def evaluate_candidate(state: ServingState):
    candidate = state["candidate_f1"]
    champion = state["champion_f1"]
    attempt = state["attempt"]

    if candidate > champion + 0.005:
        action = "promote"
        reason = "Challenger exceeds champion by the promotion margin."
        confidence = 0.96
    elif attempt < 3:
        action = "retry"
        reason = "Candidate is not good enough; another bounded training trial is allowed."
        confidence = 0.88
    else:
        action = "human_review"
        reason = "Retry budget exhausted without a clear promotion candidate."
        confidence = 0.82

    decision = TrainingDecision(
        action=action,
        model_family="xgboost",
        confidence=confidence,
        reason=reason,
        max_trials=3,
    )
    return {
        "decision": decision,
        "history": state.get("history", []) + [f"decision={decision.action}"],
    }

def promote(state: ServingState):
    return {"history": state.get("history", []) + ["PROMOTE candidate through approved MLflow action"]}

def human_review(state: ServingState):
    return {"history": state.get("history", []) + ["HUMAN REVIEW required"]}


## 6. The router is deterministic

The decision may ultimately come from an LLM, but the router consumes a validated `TrainingDecision`.


In [ ]:
def route(decision_state: ServingState) -> str:
    decision = decision_state["decision"]
    if decision.confidence < 0.70:
        return "human_review"
    return decision.action

route({"decision": TrainingDecision(
    action="retry", model_family="xgboost", confidence=0.9, reason="demo"
)})


## 7. Build the LangGraph loop

Notice the cycle: `evaluate -> retry -> train -> evaluate`. That is something a classical DAG cannot contain as a graph cycle.


In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ServingState)
builder.add_node("inspect", inspect_model)
builder.add_node("train", train_candidate)
builder.add_node("evaluate", evaluate_candidate)
builder.add_node("promote", promote)
builder.add_node("human_review", human_review)

builder.add_edge(START, "inspect")
builder.add_edge("inspect", "train")
builder.add_edge("train", "evaluate")

builder.add_conditional_edges(
    "evaluate",
    route,
    {
        "retry": "train",          # <-- graph loop
        "promote": "promote",
        "human_review": "human_review",
        "stop": END,
        "train": "train",
    },
)

builder.add_edge("promote", END)
builder.add_edge("human_review", END)

graph = builder.compile()
result = graph.invoke(initial_state)
result


In [ ]:
print("\n".join(result["history"]))


## 8. Visualize the graph

In Jupyter, LangGraph can render the compiled graph. If Mermaid rendering is unavailable in your environment, the graph still executes normally.


In [ ]:
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception as exc:
    print("Graph rendering unavailable here:", exc)
    print(graph.get_graph().draw_mermaid())


## 9. Where LangChain fits

LangChain supplies the model and tool abstractions that can live inside LangGraph nodes. A simple agent can use tools and return a Pydantic object.

A real model call is optional in this notebook because the graph mechanics are easier to understand first.


In [ ]:
# Optional real LangChain structured-output example.
# Requires a configured provider/API key.
#
# from langchain.agents import create_agent
#
# decision_agent = create_agent(
#     model="openai:gpt-5.4",
#     tools=[],
#     response_format=TrainingDecision,
#     system_prompt=(
#         "You are an ML serving controller. Choose only safe training actions. "
#         "Never promote unless the challenger clearly beats the champion."
#     ),
# )
#
# response = decision_agent.invoke({
#     "messages": [{
#         "role": "user",
#         "content": "Champion F1=.92, challenger F1=.89, attempt=1. What next?"
#     }]
# })
# response["structured_response"]


## 10. Use vLLM underneath LangChain

vLLM commonly exposes an OpenAI-compatible HTTP endpoint. LangChain can talk to that endpoint through `ChatOpenAI`, so LangGraph does not have to care whether the model is hosted by a commercial provider or your own vLLM deployment.


In [ ]:
# Optional local/private vLLM example.
# from langchain_openai import ChatOpenAI
#
# llm = ChatOpenAI(
#     model="your-served-model-name",
#     base_url="http://vllm.default.svc.cluster.local:8000/v1",
#     api_key="not-used-by-this-private-endpoint",
#     temperature=0,
# )
#
# llm.invoke("Summarize why model serving latency matters in one sentence.")


## 11. Replace the deterministic evaluation node with an LLM decision

Keep the metrics collection deterministic. Give the LLM only the compact facts it needs. Validate its output with Pydantic, then route on the typed result.


In [ ]:
def build_decision_context(state: ServingState) -> dict:
    return {
        "model": state["model_name"],
        "champion_f1": state["champion_f1"],
        "candidate_f1": state["candidate_f1"],
        "attempt": state["attempt"],
    }

# Production idea:
# facts = build_decision_context(state)
# decision = structured_llm.invoke(facts)  # returns TrainingDecision
# return {"decision": decision}

build_decision_context(result)


## 12. MLflow is the execution/registry boundary

LangGraph should not become your model registry. A graph node can call an MLflow-facing function; MLflow owns experiments, artifacts and model versions.


In [ ]:
def promote_model_in_mlflow(model_name: str, candidate_version: str) -> dict:
    # Production implementation would use MLflow client APIs.
    return {
        "model": model_name,
        "candidate_version": candidate_version,
        "status": "promotion-request-recorded",
    }

promote_model_in_mlflow("xgboost", "18")


## 13. Postgres + pgvector is knowledge, not TensorFlow weights

A useful separation:

- Iceberg/Parquet/object store = bulk training data
- PostgreSQL = structured metadata, feature/label definitions, run relationships
- pgvector = semantic retrieval across datasets/experiments/agent memory
- MLflow = model/run artifacts and registry
- TensorFlow/PyTorch = trainable model weights
- LangGraph = decision/control loop

The agent can retrieve similar historical experiments from pgvector before deciding which training DAG to launch.


In [ ]:
# Shape of the retrieval capability; replace with psycopg/SQLAlchemy + pgvector in production.
def find_similar_experiments(problem_description: str) -> list[dict]:
    return [
        {"model": "xgboost", "f1": 0.94, "learning_rate": 0.07},
        {"model": "sklearn", "f1": 0.90, "estimator": "random_forest"},
    ]

find_similar_experiments("binary churn classification with tabular customer behavior")


## 14. Observability belongs around every node

The most useful trace hierarchy is often:

`HTTP -> LangGraph run -> graph node -> LLM/tool/retrieval/MLflow -> database/model endpoint`

Prometheus answers *is the service healthy?* Langfuse/AI traces answer *why did the agent behave this way?* OpenTelemetry ties the distributed path together.


In [ ]:
from contextlib import contextmanager
from time import perf_counter

@contextmanager
def observed_step(name: str):
    started = perf_counter()
    try:
        yield
        status = "ok"
    except Exception:
        status = "error"
        raise
    finally:
        print(f"OBSERVE step={name} status={status} duration={perf_counter()-started:.4f}s")

with observed_step("model_health"):
    print(get_model_health("xgboost"))


## 15. Full production architecture

```text
Kafka / schedule / API request
          |
          v
      LangGraph  <---- Redis/Postgres checkpoint/state
          |
          +---- LangChain model/tool abstractions ----> vLLM / hosted LLM
          |
          +---- Postgres + pgvector ----> persistent knowledge
          |
          +---- FastAPI/MCP tools ----> model/data/ops capabilities
          |
          +---- MLflow/KFP training DAG ----> TensorFlow/PyTorch/sklearn
          |
          +---- Pydantic + policy + HITL ----> safe action boundary

Everything emits OTEL/Prometheus telemetry -> Grafana + Tempo/Loki + Langfuse
```

The key principle is **separation of responsibility**. LangGraph is the brain around the pipelines, not a replacement for every system underneath it.


## 16. Exercises

1. Add a `rollback` action to `TrainingDecision` and route it to a new node.
2. Add a `query_pgvector` node before training and let it influence the maximum retry count.
3. Replace `launch_training()` with an MLflow or model-serving API call.
4. Add an MCP tool wrapper around `get_model_health()`.
5. Add a Prometheus counter for graph retries and a histogram for node duration.
6. Add a checkpointer so an approval workflow can stop and resume.
7. Replace the deterministic evaluator with a LangChain structured-output agent backed by vLLM.

If you can do those seven changes, you have the core mechanics of the production architecture.
